In [0]:
from pyspark.sql.functions import col, current_timestamp, to_timestamp

# 1. Configurar credenciais
storage_account_name = "datalakeecommerce123"
storage_account_access_key = "SUA_CHAVE_AQUI_NO_DATABRICKS_SECRETS"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_access_key
)

# 2. Caminhos dos containers
path_bronze = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/ecommerce"
path_silver = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/ecommerce"

# 3. Ler dados da Camada Bronze (Delta)
df_bronze = spark.read.format("delta").load(path_bronze)

# 4. Limpeza e Tratamento dos Dados
df_silver = (df_bronze
    .dropDuplicates()  # Remove registros idênticos duplicados
    .dropna(subset=["InvoiceNo", "CustomerID"])  # Remove vendas sem identificação essencial
    .withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), "M/d/yyyy H:m"))  # Converte data para timestamp
    .withColumn("UnitPrice", col("UnitPrice").cast("double"))  # Garante tipo numérico decimal
    .withColumn("Quantity", col("Quantity").cast("integer"))   # Garante tipo inteiro
    .withColumn("data_processamento", current_timestamp())    # Coluna de auditoria
)

# 5. Salvar na Camada Silver (Delta)
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .save(path_silver)
)

print(f"Camada Silver gerada com sucesso! Total de registros limpos: {df_silver.count()}")

Camada Silver gerada com sucesso! Total de registros limpos: 401604
